# Week 19 · Notebook 02: Vertex Agent & Eval

# Requirements: pip install google-genai google-adk pandas

# ⚠️ REQUIRES: Google Cloud project

> 💰 COST WARNING: set a budget alert before running, Vertex bills per token and Agent Engine bills per session.


## What you build

Rebuild the ZoroLogistics support agent code-first with ADK, run a golden-set evaluation, and (optionally) log results to BigQuery. Dry-run mode uses a deterministic local agent loop so the eval score is reproducible without any cloud project.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root (template)
# Robust fallback: walk up until we find zoro/data.py, in case Jupyter started elsewhere.
_root = pathlib.Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "zoro" / "data.py").exists():
        sys.path.insert(0, str(_p))
        break

import os
import json
import re
import numpy as np
import pandas as pd
from zoro import data

SEED = 19
rng = np.random.default_rng(SEED)

ship = data.shipments(n=5_000, seed=42)
policies = data.policy_docs()
print("Loaded", len(ship), "shipments and", len(policies), "policy docs.")


## The agent's tools

Same support-agent behavior as Weeks 14 to 16 and Week 18: `track_shipment` (function calling) and a policy retriever. ADK wraps these as `FunctionTool` objects.


In [ ]:
def track_shipment(tracking_number: str) -> dict:
    """Return status and ETA for a ZoroLogistics shipment id."""
    row = ship[ship["shipment_id"] == str(tracking_number).strip()]
    if row.empty:
        return {"found": False, "message": "No shipment with that tracking id."}
    s = row.iloc[0]
    return {"found": True, "shipment_id": str(s["shipment_id"]),
            "status": str(s["status"]), "is_on_time": bool(s["is_on_time"])}

def check_refund_policy(amount: float) -> str:
    if amount > 500:
        return "Requires human approval."
    return "Eligible for automatic refund."

print("track_shipment('S0000001') ->", track_shipment("S0000001"))
print("check_refund_policy(750) ->", check_refund_policy(750.0))


## Build the ADK agent (code-first)

ADK is Google's open-source agent framework: `Agent(model, name, instruction, tools)`. Deploy it to Agent Engine for a hosted endpoint. The import is guarded so the notebook still runs without the SDK.


In [ ]:
ADK_AGENT = None
try:
    from google.adk.agents import Agent
    from google.adk.tools import FunctionTool
    ADK_AGENT = Agent(
        model="gemini-2.5-flash",
        name="zoro-support-agent",
        instruction=(
            "You are the ZoroLogistics support agent. Track shipments, answer policy "
            "questions, and route refunds above $500 to human approval."
        ),
        tools=[FunctionTool(track_shipment), FunctionTool(check_refund_policy)],
    )
    print("✅ ADK agent defined:", ADK_AGENT.name)
except Exception as e:  # noqa: BLE001
    print("⚠️ google-adk not installed or import failed:", type(e).__name__, e)
    print("  pip install google-adk  # then re-run; dry-run uses a local loop instead.")


## Vertex client (enterprise path)

The unified `google-genai` client targets Vertex with `vertexai=True` + project/location and application-default credentials (`gcloud auth application-default login`). Dry-run prints setup guidance.


In [ ]:
project = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
location = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
VERTEX_READY = bool(project)

if not VERTEX_READY:
    print("⚠️ Google Cloud project not set, running in DRY-RUN mode.")
    print(" gcloud auth application-default login")
    print(" gcloud config set project YOUR_GCP_PROJECT")
    print(" export GOOGLE_CLOUD_PROJECT=YOUR_GCP_PROJECT")

def vertex_client():
    from google import genai
    return genai.Client(vertexai=True, project=project, location=location)


## Golden-set evaluation

Port the Week 11 golden set and score the agent. The local loop does real tool use: a tracking question calls `track_shipment`, policy questions hit the retriever. With ADK deployed, swap in the ADK runner, the scoring stays identical.


In [ ]:
KEYWORD_DOCS = {"refund": "POL-002", "late": "POL-002", "address": "POL-001",
               "dangerous": "POL-003", "customs": "POL-004", "storage": "POL-004"}

def answer_question(question):
    q = question.lower()
    m = re.search(r"s\d{7}", q)
    if "track" in q and m:
        return json.dumps(track_shipment(m.group(0).upper()))
    doc_id = next((d for k, d in KEYWORD_DOCS.items() if k in q), None)
    if doc_id:
        for d in policies:
            if d["doc_id"] == doc_id:
                return d["text"]
    return "I could not find an answer in the shipping policies."

golden = [
    {"id": "Q1", "question": "Track shipment S0000001", "expect_contains": ["found", "S0000001"]},
    {"id": "Q2", "question": "What is the address-change fee after pickup?", "expect_contains": ["$85"]},
    {"id": "Q3", "question": "What is the storage fee for customs holds beyond 5 days?", "expect_contains": ["$40"]},
    {"id": "Q4", "question": "What documentation does a dangerous-goods shipment need?", "expect_contains": ["UN number"]},
]

eval_rows = []
for item in golden:
    ans = answer_question(item["question"])
    hits = [k for k in item["expect_contains"] if k.lower() in ans.lower()]
    passed = len(hits) == len(item["expect_contains"])
    eval_rows.append({"id": item["id"], "passed": passed, "hits": len(hits),
                       "expected": len(item["expect_contains"])})

eval_df = pd.DataFrame(eval_rows)
eval_score = eval_df["passed"].mean() if len(eval_df) else 0.0
print(eval_df.to_string(index=False))
print(f"\nEVAL_SCORE: {eval_score:.3f}")


## Optional: log results to BigQuery

Vertex's distinctive move is in-warehouse ML. Load the eval rows (or the OCR output) into BigQuery and you can later run `ML.GENERATE_TEXT` over them without leaving SQL. This cell is guarded: without a project + credentials it prints guidance instead of crashing.


In [ ]:
def log_to_bigquery(rows_df, table_id):
    from google.cloud import bigquery
    client = bigquery.Client(project=project)
    job = client.load_table_from_dataframe(rows_df, table_id)
    return job.result()

TABLE_ID = os.environ.get("BIGQUERY_EVAL_TABLE", "")
if VERTEX_READY and TABLE_ID:
    try:
        log_to_bigquery(eval_df, TABLE_ID)
        print("✅ Logged eval rows to", TABLE_ID)
    except Exception as e:  # noqa: BLE001
        print("⚠️ BigQuery write failed:", type(e).__name__, e)
else:
    print("SKIPPED BigQuery log (set GOOGLE_CLOUD_PROJECT + BIGQUERY_EVAL_TABLE and",
          "run `pip install google-cloud-bigquery`).")


## Compare with the Foundry agent (Week 18)

The product is constant; only the platform changed. Note for your Week 20 matrix: how you defined the agent (ADK vs Microsoft Agent Framework), how you evaluated it, and the cost model (Vertex per-token/per-session vs Foundry per-token + gateway).


In [ ]:
# Final number: golden-set eval score for the rebuilt agent.
print(f"EVAL_SCORE: {eval_score:.3f}")
